# 3D toric code — phase diagram in the $(h_z, h_x)$ plane

Hand-plotted phase boundary from the NQS transition estimates. Axes: $h_z$
(horizontal), $h_x$ (vertical). Two cuts: the **vertical** $h_z$ transition
(~0.21) and the **horizontal** $h_x$ transition (~1.0). The **topological** phase
is the bottom-left corner — small in *both* fields; everywhere past either line is
**trivial**. Enter each boundary's points (with error bars) in §1 and run §2.

In [ ]:
# ====================== 1 · INPUT — edit these ======================
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import patheffects as pe

# VERTICAL boundary — the h_z transition (near h_z=0.21).
#   rows: (hx, hz_c, hz_c_err)   error bar on hz_c -> HORIZONTAL (h_z is the x-axis)
BOUNDARY_V = [
    (0.0, 0.216,   0.04),
    (0.2, 0.2307,  0.03),
    (0.4, 0.2198,  0.04),
    (0.6, 0.21185, 0.03),
    (0.8, 0.2023, 0.03),
]

# HORIZONTAL boundary — the h_x transition (near h_x=1.0).
#   rows: (hz, hx_c, hx_c_err)   error bar on hx_c -> VERTICAL (h_x is the y-axis)
#   placeholder until the h_x-transition experiments finish; set to [] to hide.
BOUNDARY_H = [
    (0.0,  0.8862, 0.10),
    (0.1,  0.9762, 0.08),
    (0.2,  0.9191, 0.0904),
    (0.3,  0.9844, 0.12),
    (0.4,  0.9479, 0.0785),
    (0.5,  0.98, 0.0669),
    (0.7,  1.3089, 0.1024),
    (0.9,  1.7009, 0.1351),
    (1.0,  1.5875, 0.1202),
    (1.1,  1.7356, 0.1300),
]

# ---- plot window & style knobs ----
HZ_LIM = (0.0, 1.2)       # x-axis (h_z) range
HX_LIM = (0.0, 2.0)       # y-axis (h_x) range
# ---- generic styling (no order implied): regions Top. / Trivial 1 / Trivial 2 ----
SMOOTH = 14               # boundary-line smoothing (Gaussian sigma over interp pts; 0 = raw)
PAD    = 0.001             # axis padding (fraction of range) so edge points/error bars aren't clipped
LABELS = {"topo": "Top.", "triv1": "Trivial 1", "triv2": "Trivial 2"}

# Each style is self-contained. Both boundaries share one line style (nothing order-specific).
STYLES = {
  "A": dict(name="A · minimal grey",       topo="#6f5ea6", triv1="#ececec", triv2="#ececec",
            lc="#111111", lw=2.0, ls="-",  mk="o", ms=5,   txt="#1a1a1a", toptxt="white",   spine="#222222"),
  "B": dict(name="B · two-tone trivial",   topo="#6f5ea6", triv1="#dfe7ef", triv2="#f2e8dd",
            lc="#111111", lw=2.0, ls="-",  mk="o", ms=5,   txt="#333333", toptxt="white",   spine="#222222"),
  "C": dict(name="C · soft pastel dashed", topo="#b8abdd", triv1="#f7f7f5", triv2="#eef1f3",
            lc="#3a3a3a", lw=1.7, ls="--", mk="o", ms=4.5, txt="#444444", toptxt="#2a2140", spine="#888888"),
  "D": dict(name="D · bold deep",          topo="#4b3f7d", triv1="#e7e7e4", triv2="#e7e7e4",
            lc="#000000", lw=2.6, ls="-",  mk="s", ms=6,   txt="#111111", toptxt="white",   spine="#111111"),
}
SHOW = ["C"]   # which styles to render/save (subset of STYLES keys)

import os
FIGDIR = "../figures"   # savefig target
os.makedirs(FIGDIR, exist_ok=True)
print(f"vertical: {len(BOUNDARY_V)} pts | horizontal: {len(BOUNDARY_H)} pts | styles: {SHOW}")

## 2 · Phase diagram

In [ ]:
# ====================== 2 · PLOT — renders every style in SHOW ======================
from scipy.ndimage import gaussian_filter1d

def _interp(x, pts, ix, iy):
    q = sorted(pts, key=lambda r: r[ix])
    return np.interp(np.ravel(x), [r[ix] for r in q], [r[iy] for r in q]).reshape(np.shape(x))

xlo, xhi = HZ_LIM; ylo, yhi = HX_LIM
px = PAD*(xhi - xlo); py = PAD*(yhi - ylo)                 # padding so edge points aren't clipped
vlo_x, vhi_x = xlo - px, xhi + px; vlo_y, vhi_y = ylo - py, yhi + py
zz = np.linspace(vlo_x, vhi_x, 700); xx = np.linspace(vlo_y, vhi_y, 700)   # mesh spans padded view
ZZ, XX = np.meshgrid(zz, xx)
hzc = _interp(XX, BOUNDARY_V, 0, 1) if BOUNDARY_V else np.full_like(ZZ, xhi + 1)
hxc = _interp(ZZ, BOUNDARY_H, 0, 1) if BOUNDARY_H else np.full_like(ZZ, yhi + 1)
topo  = (ZZ < hzc) & (XX < hxc)          # topological: small in BOTH fields
triv1 = (XX >= hxc)                       # Trivial 1: above the horizontal boundary
                                          # Trivial 2: everything else (axes facecolor)
if BOUNDARY_H:
    h = sorted(BOUNDARY_H, key=lambda r: r[0])
    hz = np.array([r[0] for r in h]); hxv = np.array([r[1] for r in h]); he = np.array([r[2] for r in h])
    zs = np.linspace(hz.min(), hz.max(), 300)
    xs = gaussian_filter1d(np.interp(zs, hz, hxv), SMOOTH, mode="nearest") if SMOOTH else np.interp(zs, hz, hxv)
if BOUNDARY_V:
    v = sorted(BOUNDARY_V, key=lambda r: r[0])
    vx = [r[0] for r in v]; vz = [r[1] for r in v]; ve = [r[2] for r in v]
    zc = float(np.median(vz)); corner = float(_interp(np.array([zc]), BOUNDARY_H, 0, 1)[0]) if BOUNDARY_H else max(vx)
else:
    zc, corner = 0.21, 1.0

def draw_phase(ax, S):
    ax.set_facecolor(S["triv2"])
    ax.contourf(ZZ, XX, triv1.astype(float), levels=[0.5, 1.5], colors=[S["triv1"]], zorder=0)
    ax.contourf(ZZ, XX, topo.astype(float),  levels=[0.5, 1.5], colors=[S["topo"]],  zorder=1)
    lk = dict(color=S["lc"], lw=S["lw"], ls=S["ls"], zorder=4)
    mk = dict(fmt=S["mk"], ms=S["ms"], mfc=S["lc"], mec=S["lc"], ecolor=S["lc"],
              elinewidth=1.3, capsize=3.5, capthick=1.3, zorder=5)
    if BOUNDARY_H:
        ax.plot(zs, xs, **lk); ax.errorbar(hz, hxv, yerr=he, **mk)      # horizontal (h_x) boundary
    if BOUNDARY_V:
        ax.plot(vz + [zc], vx + [corner], **lk); ax.errorbar(vz, vx, xerr=ve, **mk)  # vertical (h_z) boundary
    tk = dict(ha="center", va="center", fontsize=14, fontweight="bold", zorder=7)
    ax.text(0.5*zc, 0.45*corner, LABELS["topo"],  color=S["toptxt"], **tk)
    ax.text(0.42*xhi, 0.88*yhi,  LABELS["triv1"], color=S["txt"], **tk)
    ax.text(0.68*xhi, 0.30*yhi,  LABELS["triv2"], color=S["txt"], **tk)
    ax.set(xlim=(vlo_x, vhi_x), ylim=(vlo_y, vhi_y))
    ax.set_xlabel("$h_z$", fontsize=14); ax.set_ylabel("$h_x$", fontsize=14)
    ax.set_title(f"3D toric code — phase diagram", fontsize=13, fontweight="bold")
    ax.tick_params(labelsize=12)
    for sp in ax.spines.values(): sp.set_edgecolor(S["spine"]); sp.set_linewidth(1.2)

for key in SHOW:
    fig, ax = plt.subplots(figsize=(8.6, 6.0))
    draw_phase(ax, STYLES[key])
    plt.tight_layout(); plt.show()
    fig.savefig(f"{FIGDIR}/phase_diagram_{key}.png", dpi=300, bbox_inches="tight")
    print(f"saved -> {FIGDIR}/phase_diagram_{key}.png")
